# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides an end-to-end guide for loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library. All dataset elements, such as record sets, fields, and columns, are referenced by their `@id` per FAIR schema best practices.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and record sets from the Clinical and Molecular Characteristics of Second Primary Colorectal Cancer dataset via `mlcroissant`. This step will fetch schema information and data previews directly from the Croissant descriptor.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant JSON-LD URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata and schema
dataset = mlc.Dataset(croissant_url)

# Print a short overview using metadata attributes
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")

## 2. Data Overview
Review available **record sets**, **fields**, and their `@id`s. We'll enumerate all record sets defined in this dataset, then show their available fields (by `@id` and descriptive name) to assist downstream selection and referencing.

In [ ]:
from mlcroissant.types.record_set import RecordSet

# List all record sets by @id and label
print("Record sets available in the dataset:")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"  - @id: {rs['@id']}, name: {rs.get('name', '<no name provided>')}")

# As a demonstration, for each record set, list its fields by @id
for rs in record_sets:
    print(f"\nFields for record set @id: {rs['@id']}")
    rs_fields = rs.get('field', [])
    # Ensure rs_fields is a list
    if isinstance(rs_fields, dict):
        rs_fields = [rs_fields]
    for f in rs_fields:
        print(f"  - @id: {f['@id']}, name: {f.get('name', '<no name provided>')}, type: {f.get('dataType', '<no type>')}")

## 3. Data Extraction
Load data from one or more specific **record set(s)** into pandas DataFrame(s) for analysis. Use the record set and field `@id`s as shown in the overview above. For this dataset, the main tabular data is commonly provided as the first or only record set. If there's more than one, you can select desired ones from the list above.

In [ ]:
# --- Specify record set(s) to load, by their @id ---
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}
for rs_id in record_set_ids:
    print(f"\nLoading data for record set @id: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            dataframes[rs_id] = pd.DataFrame(records)
            print(f"DataFrame shape: {dataframes[rs_id].shape}")
            print(f"Columns: {dataframes[rs_id].columns.tolist()}")
            display(dataframes[rs_id].head())
        else:
            print("No records found in this record set.")
    except Exception as e:
        print(f"Could not load data for {rs_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Let's perform basic EDA, referencing columns by their Croissant `@id`. This includes:
- Filtering records based on a numeric field
- Normalizing a numeric field
- Grouping data by a categorical/group field

Replace the chosen field and group IDs below with those that fit your use-case. Refer to the earlier data overview/code outputs for correct `@id`s.

In [ ]:
# -- Pick the first record set and example field IDs to demonstrate EDA --
# Update these values to match the @id's from your actual fields, as needed
first_rs_id = record_set_ids[0] if record_set_ids else None

# Example: Find numeric fields and choose one
df = dataframes.get(first_rs_id)
if df is not None:
    print(f"Numeric/float/integer columns in DataFrame {first_rs_id}:")
    numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    print(numeric_cols)

    if numeric_cols:
        numeric_field_id = numeric_cols[0]
    else:
        print("No numeric columns found!")
        numeric_field_id = None
    
    # Choose a group (category) column if available
    group_field_id = None
    for c in df.columns:
        if c != numeric_field_id and df[c].dtype == object:
            group_field_id = c
            break
    
    print(f"Selected numeric field: {numeric_field_id}")
    print(f"Selected group field: {group_field_id}")

    # Set a filter threshold for demonstration
    threshold = df[numeric_field_id].mean() if numeric_field_id else 0
    print(f"Using threshold (mean): {threshold}")

    # Filter records by numeric threshold
    if numeric_field_id is not None:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered DataFrame where {numeric_field_id} > {threshold} (first 5 rows):")
        display(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} column (first 5 rows):")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a group field if possible
        if group_field_id is not None:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame("mean").reset_index()
            print(f"Grouped means by {group_field_id}:")
            display(grouped_df)
else:
    print("No dataframes loaded to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships for the selected numeric and group fields. For example, plot the normalized numeric field distribution and means per group.

_(If running in a notebook environment, plots will be displayed inline.)_

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and numeric_field_id is not None:
    plt.figure(figsize=(7, 4))
    sns.histplot(filtered_df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id} (filtered)")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Plot group means if group field exists
    if 'grouped_df' in locals() and group_field_id is not None:
        plt.figure(figsize=(8, 4))
        sns.barplot(data=grouped_df, x=group_field_id, y='mean')
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean of {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to access, process, and visualize the FAIR^2 colorectal cancer survivor dataset using `mlcroissant`, strictly referencing all schema features by their `@id`. This approach ensures robust, machine-actionable code for interoperable data science pipelines.

**Key findings:**
- Metadata, record sets, and fields can be programmatically browsed and referenced by `@id`, enabling reproducible analysis.
- Sample data filtering, normalization, and group-level aggregation can be implemented in a few lines via `pandas` and `mlcroissant`.
- Visualizations enable rapid exploration and insight extraction from biomedical tabular data annotated in Croissant format.

For additional exploration, you can iterate over different record sets or fields by changing the relevant `@id` variables above. See the outputs in Sections 2 and 3 for available schema elements.